In [2]:
import pandas as pd
from tqdm import tqdm
import geopandas as gpd
from shapely import intersection
from shapely.geometry import Polygon
from shapely.ops import unary_union
import itertools
import numpy as np

In [3]:
SHAPE_FILE = "../../data/cb_2018_us_state_500k/cb_2018_us_state_500k.shp"

SIDE_LEN = 0.5
OUTPUT_FILE = "../../data/temperature/temp_region_area.csv"

In [4]:
REGIONS = {
    "Region 1": ["Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island", "Vermont"],
    "Region 2": ["New Jersey", "New York", "Puerto Rico", "United States Virgin Islands"],
    "Region 3": ["Delaware", "District of Columbia", "Maryland", "Pennsylvania", "Virginia", "West Virginia"],
    "Region 4": ["Alabama", "Florida", "Georgia", "Kentucky", "Mississippi", "North Carolina", "South Carolina", "Tennessee"],
    "Region 5": ["Illinois", "Indiana", "Michigan", "Minnesota", "Ohio", "Wisconsin"],
    "Region 6": ["Arkansas", "Louisiana", "New Mexico", "Oklahoma", "Texas"],
    "Region 7": ["Iowa", "Kansas", "Missouri", "Nebraska"],
    "Region 8": ["Colorado", "Montana", "North Dakota", "South Dakota", "Utah", "Wyoming"],
    "Region 9": ["Arizona", "California", "Hawaii", "Nevada", "American Samoa", "Commonwealth of the Northern Mariana Islands", "Guam"],
    "Region 10": ["Alaska", "Idaho", "Oregon", "Washington"]
}

In [5]:
# creates a square centered at point
d = SIDE_LEN / 2
coord_to_square = lambda x: Polygon([
            (x[0]-d, x[1]-d),
            (x[0]+d, x[1]-d),
            (x[0]+d, x[1]+d),
            (x[0]-d, x[1]+d)
        ])

# list of all squares
lon_range = np.arange(-360+d, 0, SIDE_LEN).tolist()
lat_range = np.arange(-90+d, 90, SIDE_LEN).tolist()
coords = list(itertools.product(lon_range, lat_range))
coord_squares = list(map(coord_to_square, coords))

In [6]:
# states geometry
states_gdf = gpd.read_file(SHAPE_FILE)
states_gdf.head()

,STATEFP,STATENS,AFFGEOID,GEOID,STUSPS,NAME,LSAD,ALAND,AWATER,geometry
0,28,01779790,0400000US28,28,MS,Mississippi,00,121533519481,3926919758,"MULTIPOLYGON (((-88.50297 30.21524, -88.49176 ..."
1,37,01027616,0400000US37,37,NC,North Carolina,00,125923656064,13466071395,"MULTIPOLYGON (((-75.72681 35.93584, -75.71827 ..."
2,40,01102857,0400000US40,40,OK,Oklahoma,00,177662925723,3374587997,"POLYGON ((-103.00256 36.52659, -103.00219 36.6..."
3,51,01779803,0400000US51,51,VA,Virginia,00,102257717110,8528531774,"MULTIPOLYGON (((-75.74241 37.80835, -75.74151 ..."
4,54,01779805,0400000US54,54,WV,West Virginia,00,62266474513,489028543,"POLYGON ((-82.6432 38.16909, -82.643 38.16956,..."


In [7]:
# combine states into regions
region_data = []
for region, states in REGIONS.items():
    region_gdf = states_gdf[states_gdf.NAME.isin(states)]
    region_data.append({
        "region": region,
        "geometry": unary_union(region_gdf.geometry)
    })
regions_gdf = pd.DataFrame(region_data)
regions_gdf

,region,geometry
0,Region 1,"MULTIPOLYGON (((-73.62512699999999 40.981213, ..."
1,Region 2,"MULTIPOLYGON (((-75.547197 39.640527999999996,..."
2,Region 3,"MULTIPOLYGON (((-76.417362 37.591656, -76.4191..."
3,Region 4,"MULTIPOLYGON (((-82.864144 24.624733, -82.8643..."
4,Region 5,"MULTIPOLYGON (((-91.507269 40.209337999999995,..."
5,Region 6,"MULTIPOLYGON (((-97.29805499999999 26.561884, ..."
6,Region 7,"POLYGON ((-91.422055931802 40.3759712737879, -..."
7,Region 8,"POLYGON ((-104.023383 41.001886999999996, -104..."
8,Region 9,MULTIPOLYGON (((-168.144004 -14.54602199999999...
9,Region 10,"MULTIPOLYGON (((-179.130013 51.290423, -179.12..."


In [8]:
# calculate percent of square in region if overlaps at all
us_coord_squares = []
regions_gdf_rows = list(regions_gdf.itertuples(index = False))
for coord, coord_square in tqdm(zip(coords, coord_squares), total = len(coords)):
    # percent of square in region for each region
    region_info = {
        region: intersection(coord_square, region_shape).area / SIDE_LEN**2
        for region, region_shape in regions_gdf_rows
    }

    if sum(region_info.values()): # square has any overlap with US
        region_info["coords"] = coord
        us_coord_squares.append(region_info)

df_region_area = pd.DataFrame(us_coord_squares).set_index("coords")
df_region_area

100%|██████████| 259200/259200 [01:24<00:00, 3053.44it/s] 


,Region 1,Region 2,Region 3,Region 4,Region 5,Region 6,Region 7,Region 8,Region 9,Region 10
coords,,,,,,,,,,
"(-179.25, 51.25)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.018633
"(-179.25, 51.75)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.001345
"(-178.75, 51.25)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.017087
"(-178.75, 51.75)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.054108
"(-178.25, 28.25)",0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000962,0.000000
...,...,...,...,...,...,...,...,...,...,...
"(-65.75, 17.75)",0.0,0.010819,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
"(-65.75, 18.25)",0.0,0.507078,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
"(-65.25, 18.25)",0.0,0.053583,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000


In [9]:
df_region_area.to_csv(OUTPUT_FILE)